In [19]:
import pandas as pd
import csv

# File paths
RAW_DATA_PATH = 'data/amazon_reviews.csv'
PROCESSED_DATA_PATH = 'data/processed_reviews.csv'

In [ ]:
# Load the raw dataset
df_raw = pd.read_csv(RAW_DATA_PATH)

# Combine Summary and Text for better context
df_raw['full_text'] = df_raw['Summary'].fillna('') + " " + df_raw['Text'].fillna('')

# Select relevant columns and rename to snake_case for consistency
df = df_raw[['Id', 'full_text', 'Score']].copy()
df.columns = ['id', 'full_text', 'score']

print(f"Initial records loaded: {len(df)}")


Initial records loaded: 568454


In [21]:
df.head()

,id,full_text,score
0,1,Good Quality Dog Food I have bought several of...,5
1,2,Not as Advertised Product arrived labeled as J...,1
2,3,"""Delight"" says it all This is a confection tha...",4
3,4,Cough Medicine If you are looking for the secr...,2
4,5,Great taffy Great taffy at a great price. The...,5


In [22]:
# Map numerical scores to sentiment: -1 (Neg), 0 (Neu), 1 (Pos)
def map_sentiment(score):
    if score <= 2: return -1
    if score == 3: return 0
    return 1

df['sentiment'] = df['score'].apply(map_sentiment)

# Add a machine learning label (0, 1, 2) for model compatibility
df['label'] = df['sentiment'] + 1

# Remove duplicates and rows with missing text
df = df.drop_duplicates(subset=['full_text']).dropna()

print("Distribution before balancing:")
print(df['sentiment'].value_counts().sort_index())

Distribution before balancing:
sentiment
-1     57346
 0     29837
 1    307787
Name: count, dtype: int64


In [23]:
# Identify the size of the smallest class to ensure a perfect balance
min_class_size = df['sentiment'].value_counts().min()

print(f"Balancing dataset to {min_class_size} samples per class...")

# Create balanced subsets
df_neg = df[df['sentiment'] == -1].sample(min_class_size, random_state=42)
df_neu = df[df['sentiment'] == 0].sample(min_class_size, random_state=42)
df_pos = df[df['sentiment'] == 1].sample(min_class_size, random_state=42)

# Combine and shuffle the final dataset
df_balanced = pd.concat([df_neg, df_neu, df_pos]).sample(frac=1, random_state=42).reset_index(drop=True)

print("-" * 30)
print(f"Final balanced dataset size: {len(df_balanced)}")
print(df_balanced['sentiment'].value_counts())

Balancing dataset to 29837 samples per class...
------------------------------
Final balanced dataset size: 89511
sentiment
 0    29837
 1    29837
-1    29837
Name: count, dtype: int64


In [ ]:
# Save to a standard CSV format (comma-separated)
# quoting=csv.QUOTE_MINIMAL will wrap text in double quotes only if it contains commas

df_balanced.to_csv(
    PROCESSED_DATA_PATH, 
    index=False, 
    sep=',',            # Standard comma separator
    quoting=csv.QUOTE_MINIMAL, 
    quotechar='"',      # Standard double quote
    encoding='utf-8'
)

print(f"Data exported to classic CSV: {PROCESSED_DATA_PATH}")
# Displaying a raw string example of how it looks internally
print(df_balanced.head(2).to_csv(index=False, quoting=csv.QUOTE_MINIMAL))

Data exported to classic CSV: data/processed_reviews.csv
id,full_text,score,sentiment,label
497575,"It's not equal amounts of each flavor... Just so you know...there was only one blueberry flavored lollipop in the entire container...only 3 mango flavored lollipop and just a few of the other flavors... most of the flavors that were plentiful in the container were razzaberry, and strawberry. I would prefer equal amounts of each flavor.",3,0,1
333710,"Not A Gingerbread Man These do taste very good. However, they are not gingerbread men as you would expect from the picture. They are medium/large round cookies. In that respect I was disappointed because I wanted to give my kids Gingerbread Men for Christmas.",4,1,2

